# 入力の途中で候補を出す・打ち間違いを直す（サジェストと「もしかして」）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_search/search_suggest.ipynb)

NLP クックブック 19-18 ／ 検索

検索窓に数文字入れた時点で候補を出したい。打ち間違いや、ひらがなで入力された語にも対応したい。

In [1]:
!pip install -q rapidfuzz fugashi unidic-lite jaconv

In [2]:
import bisect
import jaconv
from fugashi import Tagger
from rapidfuzz import fuzz, process

# 候補の語と人気度 (検索回数など)
terms = {"自然言語処理": 950, "自然言語理解": 120, "自然言語生成": 200, "機械翻訳": 800, "機械学習": 990, "形態素解析": 400, "係り受け解析": 150,
         "固有表現抽出": 300, "感情分析": 350, "質問応答": 280, "文法誤り訂正": 180, "大規模言語モデル": 700, "検索拡張生成": 260, "言語モデル": 500}

tagger = Tagger()
def reading(word):
    return jaconv.kata2hira("".join(w.feature.kana or w.surface for w in tagger(word)))

# 表記と読みの両方を、前方一致で引けるように並べておく
entries = sorted([(t, t) for t in terms] + [(reading(t), t) for t in terms])
keys = [k for k, _ in entries]

def suggest(prefix, k=5):
    """前方一致する候補を、人気度の高い順に返す (二分探索なので、候補が数百万あっても速い)"""
    prefix = jaconv.kata2hira(prefix)
    lo = bisect.bisect_left(keys, prefix)
    hi = bisect.bisect_right(keys, prefix + "￿")
    found = {t for _, t in entries[lo:hi]}
    return sorted(found, key=lambda t: -terms[t])[:k]

for prefix in ["自然", "しぜんげんごし", "機械", "きかい", "けい", "ぶんぽう"]:
    print(prefix, "→", suggest(prefix))

自然 → ['自然言語処理', '自然言語生成', '自然言語理解']
しぜんげんごし → ['自然言語処理']
機械 → ['機械学習', '機械翻訳']
きかい → ['機械学習', '機械翻訳']
けい → ['形態素解析']
ぶんぽう → ['文法誤り訂正']


In [3]:
# 前方一致で見つからないときは、編集距離の近い語を「もしかして」として出す
def did_you_mean(query, limit=2, threshold=60):
    by_surface = process.extract(query, list(terms), scorer=fuzz.ratio, limit=limit)
    by_reading = process.extract(reading(query), {t: reading(t) for t in terms}, scorer=fuzz.ratio, limit=limit)
    best = {}
    for name, score in [(m[0], m[1]) for m in by_surface] + [(m[2], m[1]) for m in by_reading]:
        best[name] = max(best.get(name, 0), score)
    return [(t, round(s)) for t, s in sorted(best.items(), key=lambda x: -x[1]) if s >= threshold][:limit]

for q in ["自然原語処理", "機会翻訳", "けいたいそかいせき", "固有表現注出", "大規模言語モデル", "らーめん"]:
    hits = suggest(q, 1)
    print(q, "→", hits if hits else f"もしかして: {did_you_mean(q)}")

自然原語処理 → もしかして: [('自然言語処理', 100), ('自然言語理解', 78)]
機会翻訳 → もしかして: [('機械翻訳', 100)]
けいたいそかいせき → ['形態素解析']
固有表現注出 → もしかして: [('固有表現抽出', 85)]
大規模言語モデル → もしかして: [('大規模言語モデル', 100), ('言語モデル', 77)]
らーめん → もしかして: []


- サジェストは「前方一致 + 人気順」が基本。候補をソートしておけば二分探索で引ける。大規模にはトライ木や、検索エンジンの補完機能を使う。
- 日本語では、変換前のひらがなやローマ字で入力されることが多い。読みでも引けるようにしておくと、「しぜんげんごし」の時点で候補を出せる。
- 「機会翻訳」のような同音の誤変換は、表記の編集距離では遠くても、読みに直すと完全に一致する。表記と読みの両方で照合するのが有効。
- しきい値未満なら何も出さない。無関係な語を「もしかして」と出すと、かえって邪魔になる。